# 02 · Собрать и проверить учебный online-трекер
Лекция 7. Центральная работа — Kalman, IoU, global assignment с gate/unmatched, lifecycle, матрицы стоимостей и самостоятельная сборка шага трекера. Семь TODO; работа выполняется на CPU. Вычисления занимают секунды/минуты, основное время — реализация и анализ.

Используем **учебное состояние 4D `[x,y,vx,vy]`**, фиксированные размеры рамки, EMA-дескриптор и одно назначение. Это не полный SORT 7D и не DeepSORT 8D: нет его cascade, обученной CNN и gallery descriptors.

Синтетика дает точные identity/координаты, независимые sequence seeds, пересечения, ложные наблюдения и пропуски. Отдельный стресс-тест меняет только appearance на validation после фиксации веса. Реальный видеопример пропускает MOG2-компоненты через учебный motion-only трекер; он не имеет GT и не участвует в выборе параметров.

Пакеты: numpy, scipy, matplotlib, opencv-python-headless. Внешнее видео 8.1 МБ загружается в временный кэш при первом запуске с видимым ходом и SHA-256. Загрузите notebook в Colab или откройте в Jupyter. Исходный TODO notebook намеренно останавливается до реализации.

In [ ]:
# Запускайте первой. В Colab сохраняем штатные PyTorch/CUDA и ipywidgets.
import importlib.util, subprocess, sys
if sys.version_info < (3, 10):
    raise RuntimeError('Нужен Python 3.10 или новее.')
_packages = {'numpy': 'numpy>=1.26,<3', 'matplotlib': 'matplotlib>=3.8,<4', 'scipy': 'scipy>=1.11,<2', 'cv2': 'opencv-python-headless==4.12.0.88'}
_missing = [spec for module, spec in _packages.items()
            if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_missing])
    raise RuntimeError('Пакеты установлены. Перезапустите среду выполнения и выполните ноутбук сверху.')
import numpy, matplotlib
if not (1 <= int(numpy.__version__.split('.')[0]) < 3):
    raise RuntimeError('Нужен NumPy 1.x или 2.x; используйте отдельное окружение.')
if int(matplotlib.__version__.split('.')[0]) != 3:
    raise RuntimeError('Нужен Matplotlib 3.x; используйте отдельное окружение.')
_versions = {'python': sys.version.split()[0], 'numpy': numpy.__version__,
             'matplotlib': matplotlib.__version__}
if 'scipy' in _packages:
    import scipy
    if int(scipy.__version__.split('.')[0]) != 1:
        raise RuntimeError('Нужен SciPy 1.x; используйте отдельное окружение.')
    _versions['scipy'] = scipy.__version__
if 'cv2' in _packages:
    import cv2
    _versions['opencv'] = cv2.__version__
    _required_features = ()
    _absent = [name for name in _required_features if not hasattr(cv2, name)]
    if _absent:
        raise RuntimeError('В OpenCV нет ' + ', '.join(_absent) +
                           '. Установите один вариант OpenCV 4.12.0.88 в отдельной среде и перезапустите ядро.')
if 'ipywidgets' in _packages:
    import ipywidgets
    _versions['ipywidgets'] = ipywidgets.__version__
    if int(ipywidgets.__version__.split('.')[0]) not in (7, 8):
        raise RuntimeError('Нужен ipywidgets 7.x или 8.x; в Colab 2026.07 оставьте штатный 7.7.1.')
    try:
        from google.colab import output
        output.enable_custom_widget_manager()
    except ImportError:
        pass
print('Окружение:', _versions)
print('Вычисления этой лабораторной: CPU; GPU не требуется.')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Ellipse
from scipy.optimize import linear_sum_assignment
from pathlib import Path
import time,json,csv,itertools
plt.rcParams.update({'figure.dpi':110,'font.size':10})
np.set_printoptions(precision=4,suppress=True)

## Гипотеза до вычислений
**Заполните:** почему appearance должна уменьшить смены ID в пересечении; при каком сбое ожидаете ухудшение; какой график опровергнет гипотезу.

Фиксированный протокол: val sequences seeds11–14, test 21–24; по 48 кадров, 3 объекта. Детекции, шум, пропуски и признаки совпадают между вариантами. Меняется только appearance weight 0/.25/.5/.75; gate 16, max_age4, n_init2, Q, R и размеры bbox фиксированы. Нет обучения CNN; движение и внешность уже представлены управляемыми измерениями.

Выбор: минимальная суммарная частота смен присвоенного track ID у одного GT ID между последовательными **наблюдениями**, включая разрывы; ties → меньший weight. После val параметр записывается, затем оценивается test.

Это учебная continuity-метрика, **не официальная MOTChallenge IDSW/MOTA/IDF1/HOTA**. Дополнительно считаем track purity, RMSE и время tracker/evaluator. Fragmentation здесь — сумма $(\#\text{разных track ID для GT}-1)$ по наблюдавшимся GT; знаменатель `observed_gt_ids` — число таких GT. `false_confirmed` — число треков, хоть раз подтверждённых, но ни разу не получивших истинного наблюдения; знаменатель `confirmed_tracks` — число когда-либо подтверждённых треков. Эти учебные диагностики используют GT только в оценщике и не заменяют MOT-метрики. Ложные детекции исключены из continuity, purity и RMSE; evaluator учитывает tentative присвоения.

## TODO 1 · Общие predict/update
Реализуйте линейный фильтр через матричные операции. Используйте Joseph form covariance и solve.

In [ ]:
def predict(x,P,F,Q):
    """Return F@x and F@P@F.T+Q; do not mutate arguments."""
    raise NotImplementedError('TODO 1a predict')

def update(x,P,z,H,R):
    """Return updated mean/covariance. Use solve (not inverse), Joseph covariance
    (I-KH)P(I-KH)^T + KRK^T; symmetrize numerical roundoff."""
    raise NotImplementedError('TODO 1b update')

In [ ]:
x=np.array([10.,2.]);P=np.diag([4.,1.]);F=np.array([[1.,1.],[0.,1.]])
xp,Pp=predict(x,P,F,np.diag([1.,.25]));assert np.allclose(xp,[12,2]) and np.allclose(Pp,[[6,1],[1,1.25]])
xu,Pu=update(xp,Pp,np.array([13.]),np.array([[1.,0.]]),np.array([[4.]]))
assert np.allclose(xu,[12.6,2.1]) and np.allclose(Pu,[[2.4,.4],[.4,1.15]])
assert np.allclose(x,[10,2]) and np.allclose(P,np.diag([4,1]))
for _ in range(30):
 xp,Pp=predict(xu,Pu,F,np.diag([1.,.25]));xu,Pu=update(xp,Pp,np.array([13.]),np.array([[1.,0.]]),np.array([[4.]]))
 assert np.allclose(Pu,Pu.T) and np.linalg.eigvalsh(Pu).min()>-1e-10
print('Kalman open tests passed')

## TODO 2 · IoU для всех пар
Векторизуйте NxM, обработайте пустые массивы и вырожденные рамки.

In [ ]:
def pairwise_iou(a,b):
    """Finite Nx4/Mx4 xyxy continuous boxes; empty inputs allowed.
    Zero-area union => IoU0. Reversed coordinates/nonfinite => ValueError."""
    raise NotImplementedError('TODO 2 IoU')

In [ ]:
assert np.allclose(pairwise_iou([[0,0,2,2]],[[1,0,3,2]]),[[1/3]])
assert pairwise_iou(np.empty((0,4)),[[0,0,1,1]]).shape==(0,1)
assert pairwise_iou([[0,0,0,0]],[[0,0,0,0]])[0,0]==0
try:pairwise_iou([[2,0,1,1]],[[0,0,1,1]])
except ValueError:pass
else:raise AssertionError('reversed box')
print('IoU open tests passed')

## TODO 3 · Назначение с запрещёнными рёбрами и отказом
Постройте расширенную квадратную матрицу с отдельными dummy-вершинами. Gate действует до solver; цена отказа u/2 на endpoint.

In [ ]:
def gated_assignment(cost,allowed,unmatched_cost):
    """Finite nonnegative NxM cost; matching-shape bool allowed. u>=0 finite.
    Minimize sum matched costs + u/2 for each unmatched track/detection.
    Encode prohibited edges and dummies BEFORE linear_sum_assignment.
    Return (pairs:list[(track,det)],unmatched_tracks:list,unmatched_dets:list).
    Empty inputs are valid. Do not force all real objects to match.
    Exact ties may have multiple optimal solutions; tests avoid ambiguity."""
    raise NotImplementedError('TODO 3 gated assignment with dummies')

In [ ]:
p,ut,ud=gated_assignment([[.1,.2],[.11,.05]],[[True,True],[True,False]],3)
assert set(p)=={(0,1),(1,0)} and ut==ud==[]
p,ut,ud=gated_assignment([[.1]],[[False]],1);assert p==[] and ut==[0] and ud==[0]
p,ut,ud=gated_assignment(np.empty((0,2)),np.empty((0,2),bool),1);assert p==[] and ut==[] and ud==[0,1]
p,ut,ud=gated_assignment([[.1,.9,.8]],[[True,True,True]],.5);assert p==[(0,0)] and ud==[1,2]
print('Assignment open tests passed')

## TODO 4 · Жизненный цикл
Tentative удаляется при первом пропуске, confirmed переживает ровно max_age пропусков; deleted терминален.

In [ ]:
def lifecycle(status,hits,misses,matched,n_init,max_age):
    """status: tentative/confirmed/deleted; new tracks already have hits1.
    Deleted is terminal. Match increments hits, resets misses, confirms at n_init.
    Miss increments misses; tentative deletes immediately; confirmed deletes
    only if misses>max_age. Return (status,hits,misses). Validate state/counts."""
    raise NotImplementedError('TODO 4 lifecycle')

In [ ]:
assert lifecycle('tentative',1,0,True,2,3)==('confirmed',2,0)
assert lifecycle('tentative',1,0,False,2,3)==('deleted',1,1)
assert lifecycle('confirmed',4,2,False,2,3)==('confirmed',4,3)
assert lifecycle('confirmed',4,3,False,2,3)==('deleted',4,4)
assert lifecycle('confirmed',4,3,True,2,3)==('confirmed',5,0)
print('Lifecycle open tests passed')

## Параметры учебной модели
Фиксированные F, H, Q, R и gate применяются одинаково ко всем вариантам.

In [ ]:
# Учебная 4D-модель, размеры рамок фиксированы. Это не реализация SORT/DeepSORT.
F=np.array([[1.,0,1,0],[0,1,0,1],[0,0,1,0],[0,0,0,1]])
H=np.array([[1.,0,0,0],[0,1.,0,0]])
Q=.08*np.array([[.25,0,.5,0],[0,.25,0,.5],[.5,0,1,0],[0,.5,0,1]])
R=np.eye(2)*1.15**2
GATE=16.;N_INIT=2;MAX_AGE=4;UNMATCHED=1.

## TODO 5 · Матрицы ассоциации
Постройте Mahalanobis², IoU, geometry, appearance, итоговую стоимость и независимый от внешности gate. Проверьте пустые входы и запрещённую хорошую appearance-пару.

In [ ]:
def association_matrices(tracks,z,appearance,boxes,weight):
    """Return dict with motion2, iou, appearance_cost, geometry, cost, allowed.
    tracks have already passed predict; use x/P/appearance, measurement H/R.
    motion2 = (z-Hx)^T (HPH^T+R)^(-1) (z-Hx), computed with solve.
    geometry=.75*min(motion2/GATE,1)+.25*(1-IoU).
    appearance_cost=(1-cosine)/2; cost=(1-weight)*geometry+weight*appearance_cost.
    Gate is motion2<=GATE and never depends on appearance or weight.
    All output matrices must have shape (number of tracks, number of detections),
    including when either side is empty. Do not read GT fields."""
    raise NotImplementedError('TODO 5 association matrices')

In [ ]:
sample_track={'x':np.array([0.,0.,0.,0.]),'P':np.eye(4),
              'appearance':np.array([1.,0.,0.,0.,0.,0.,0.,0.])}
sample_z=np.array([[1.,0.],[20.,0.]])
sample_app=np.tile(sample_track['appearance'],(2,1))
sample_boxes=np.c_[sample_z[:,0]-1.5,sample_z[:,1]-2,sample_z[:,0]+1.5,sample_z[:,1]+2]
terms=association_matrices([sample_track],sample_z,sample_app,sample_boxes,.25)
assert terms['cost'].shape==(1,2) and terms['allowed'].tolist()==[[True,False]]
assert np.isclose(terms['motion2'][0,0],1/(1+R[0,0]))
assert np.isclose(terms['cost'][0,0],.75*terms['geometry'][0,0]+.25*terms['appearance_cost'][0,0])
assert association_matrices([],sample_z,sample_app,sample_boxes,.5)['cost'].shape==(0,2)
assert association_matrices([sample_track],np.empty((0,2)),np.empty((0,8)),np.empty((0,4)),.5)['cost'].shape==(1,0)
assert not association_matrices([sample_track],sample_z,sample_app,sample_boxes,1.)['allowed'][0,1]
print('Association matrices open tests passed')

## Одинаковые наблюдения для всех вариантов
GT находится в отдельных полях словаря и используется только оценщиком. Tracker получает z, appearance, boxes. Перемешивание detections на каждом кадре не дает восстановить ID из порядка. Систематические пропуски объекта 1 в кадрах 21–23 и объекта 2 в 31–32 проверяют predict без update. Две близкие ложные детекции в соседних кадрах создают отдельный сценарий для ложного подтверждения; оценщик распознаёт их по GT=-1, трекер эту метку не получает.

In [ ]:
def unit(x):
 x=np.asarray(x,float);return x/np.maximum(np.linalg.norm(x,axis=-1,keepdims=True),1e-12)

def make_sequence(seed,T=48):
 rng=np.random.default_rng(seed);prototypes=unit(rng.normal(size=(3,8)))
 seq=[]
 for t in range(T):
  true=np.array([[-12+.52*t,.5*np.sin(t/7)],[12-.52*t,-.5*np.sin(t/7)],[.3*np.sin(t/5),-12+.52*t]])
  detections=[];features=[];ids=[]
  for i in range(3):
   if (i==1 and 21<=t<=23) or (i==2 and 31<=t<=32):continue
   if rng.random()<.04:continue
   detections.append(true[i]+rng.normal(0,1.15,2));features.append(unit(prototypes[i]+rng.normal(0,.17,8)));ids.append(i)
  if t%13 in (7,8):
   # Two nearby false positives can confirm a short-lived false track.
   detections.append(np.array([8.+.1*(t%13),-8.])+rng.normal(0,.15,2))
   features.append(unit(np.ones(8)));ids.append(-1)
  # Shuffle observations: ordering must not reveal identity.
  order=rng.permutation(len(ids));z=np.array(detections,float).reshape(-1,2)[order];app=np.array(features,float).reshape(-1,8)[order]
  boxes=np.c_[z[:,0]-1.5,z[:,1]-2,z[:,0]+1.5,z[:,1]+2]
  seq.append({'z':z,'appearance':app,'boxes':boxes,'gt_ids':np.array(ids)[order],'true_positions':true})
 return seq

VAL_SEEDS=[11,12,13,14];TEST_SEEDS=[21,22,23,24]
assert set(VAL_SEEDS).isdisjoint(TEST_SEEDS)
val_sequences={s:make_sequence(s) for s in VAL_SEEDS}
# Only evaluator reads gt_ids and true_positions. Tracker receives other fields.
print('val sequences:',len(val_sequences),'frames each:',len(val_sequences[11]))

def stress_appearance(sequence):
 """Fixed val-only intervention: swap features of GT 0/1 at frames 18..29.
 GT labels are used here only to construct observations, never by tracker.step.
 Positions, boxes, detections and GT remain identical to the standard sequence.
 """
 stressed=[]
 for t,frame in enumerate(sequence):
  changed=dict(frame);changed['appearance']=frame['appearance'].copy()
  if 18<=t<=29:
   left=np.flatnonzero(frame['gt_ids']==0);right=np.flatnonzero(frame['gt_ids']==1)
   if len(left)==1 and len(right)==1:
    changed['appearance'][left[0]]=frame['appearance'][right[0]]
    changed['appearance'][right[0]]=frame['appearance'][left[0]]
  stressed.append(changed)
 return stressed

## TODO 6 · Шаг online-трекера и оценщик
Соберите `TeachingTracker.step` самостоятельно: predict всем → матрицы стоимости и gate → глобальное назначение/отказ → update/miss → удаление → новые tentative ID. Сохраните численный trace решения для ручного разбора. Размер IoU bbox фиксирован и не оценивается фильтром.

Geometry cost=.75·min(Mahalanobis²/16, 1)+.25·(1−IoU). Appearance cost=(1−cosine)/2. Итог=(1−weight)·geometry+weight·appearance. Gate разрешает Mahalanobis²≤16 независимо от weight. Это выбранная учебная стоимость, а не точная стоимость DeepSORT.

In [ ]:
class TeachingTracker:
 def __init__(self,weight):
  self.weight=weight;self.tracks=[];self.next_id=0;self.last_decision={}
 def step(self,z,appearance,boxes):
  """TODO 6: one online step. Inputs are only detections, never GT.
  1. Predict each live track with F/Q.
  2. Call association_matrices and gated_assignment; save pre-update motion2,
     IoU, appearance_cost, geometry, cost, allowed, track IDs, pairs and both
     unmatched sets in last_decision.
  3. Update matched tracks (state, lifecycle, appearance EMA .8/.2).
  4. Age unmatched tracks, remove deleted, birth one tentative track per
     unmatched detection with P=diag(2,2,4,4) and velocity zero.
  Return (assigned_ids[M], filtered_positions[M,2]); every detection receives
  exactly one ID, including tentative births. Preserve monotonically growing IDs.
  last_decision must be JSON-serializable for the val audit."""
  raise NotImplementedError('TODO 6 multi-frame tracker step')

def evaluate(sequence,weight):
 tracker=TeachingTracker(weight);last={};changes=0;comparisons=0;errors=[];trace=[];groups={}
 gt_to_tracks={};true_assigned_tracks=set();ever_confirmed=set();start=time.perf_counter()
 for t,frame in enumerate(sequence):
  assigned,positions=tracker.step(frame['z'],frame['appearance'],frame['boxes'])
  ever_confirmed.update(tr['id'] for tr in tracker.tracks if tr['status']=='confirmed')
  for j,(gid,tid) in enumerate(zip(frame['gt_ids'],assigned)):
   if gid<0:continue
   gid=int(gid);tid=int(tid);true_assigned_tracks.add(tid)
   gt_to_tracks.setdefault(gid,set()).add(tid)
   if gid in last:comparisons+=1;changes+=int(last[gid]!=tid)
   last[gid]=tid;groups.setdefault(tid,[]).append(gid)
   errors.append(float(np.sum((positions[j]-frame['true_positions'][gid])**2)))
   trace.append({'frame':t,'gt_id':gid,'track_id':tid,'x':float(positions[j,0]),'y':float(positions[j,1])})
 # Fragmentation counts extra IDs for each observed GT. It is not official Frag.
 fragments=sum(max(0,len(tids)-1) for tids in gt_to_tracks.values())
 false_confirmed=len(ever_confirmed-true_assigned_tracks)
 purity=sum(max(ids.count(k) for k in set(ids)) for ids in groups.values())/sum(map(len,groups.values()))
 return {'weight':weight,'changes':changes,'comparisons':comparisons,'change_rate':changes/comparisons,
         'purity':purity,'position_rmse':float(np.sqrt(np.mean(errors))),
         'fragments':fragments,'observed_gt_ids':len(gt_to_tracks),'fragments_per_gt':fragments/len(gt_to_tracks),
         'false_confirmed':false_confirmed,'confirmed_tracks':len(ever_confirmed),
         'false_confirmed_rate':false_confirmed/max(len(ever_confirmed),1),
         'created_tracks':tracker.next_id,'seconds':time.perf_counter()-start},trace

In [ ]:
track=TeachingTracker(.5);a=np.array([[1.,0,0,0,0,0,0,0],[0,1.,0,0,0,0,0,0]])
z=np.array([[0.,0.],[10.,0.]])
boxes=np.c_[z[:,0]-1.5,z[:,1]-2,z[:,0]+1.5,z[:,1]+2]
ids,positions=track.step(z,a,boxes)
assert ids.tolist()==[0,1] and positions.shape==(2,2) and track.next_id==2
assert [tr['status'] for tr in track.tracks]==['tentative','tentative']
assert track.last_decision['track_ids_before']==[] and track.last_decision['unmatched_dets']==[0,1]
z2=np.array([[.5,0.],[9.5,0.]])
boxes2=np.c_[z2[:,0]-1.5,z2[:,1]-2,z2[:,0]+1.5,z2[:,1]+2]
ids,_=track.step(z2,a,boxes2)
assert ids.tolist()==[0,1] and [tr['status'] for tr in track.tracks]==['confirmed','confirmed']
assert track.last_decision['pairs']==[[0,0],[1,1]]
assert set(('motion2','iou','appearance_cost','geometry','cost','allowed')).issubset(track.last_decision)
for _ in range(MAX_AGE):track.step(np.empty((0,2)),np.empty((0,8)),np.empty((0,4)))
assert len(track.tracks)==2 and all(tr['misses']==MAX_AGE for tr in track.tracks)
track.step(np.empty((0,2)),np.empty((0,8)),np.empty((0,4)))
assert not track.tracks
new_id,_=track.step(z[:1],a[:1],boxes[:1]);assert new_id.tolist()==[2]
json.dumps(track.last_decision)
print('Multi-frame tracker open tests passed')

## TODO 7 · Собственные контрпримеры
Создайте минимум два новых примера с ручным прогнозом. Один должен проверять интеграцию нескольких шагов трекера; другой — численное граничное условие.

In [ ]:
# TODO 7: два новых примера с ожидаемым ответом ДО исполнения.
# Один — интеграционный (несколько кадров: predict/assignment/lifecycle/ID).
# Другой — covariance, gate, IoU, unmatched или пустой вход.
# Добавьте assert и объяснение, какая ошибка будет обнаружена.
raise NotImplementedError('TODO 7 own counterexamples')

## Val: baseline и одна управляемая причина
Выведите результаты всех четырех seeds отдельно и pooled change rate. Не выбирайте seed по красивому результату. Сохраните гипотезу и объясните выбранный weight перед переходом к test. `selection_locked.json` фиксирует решение до создания test observations.

In [ ]:
RESULTS=Path('results');RESULTS.mkdir(exist_ok=True)
WEIGHTS=[0.,.25,.5,.75]
val_rows=[]
for weight in WEIGHTS:
 for seed,sequence in val_sequences.items():
  row,_=evaluate(sequence,weight);val_rows.append({'split':'val','seed':seed,**row})
def summarize(rows,weight):
 rows=[r for r in rows if r['weight']==weight]
 return {'weight':weight,'changes':sum(r['changes'] for r in rows),'comparisons':sum(r['comparisons'] for r in rows),'change_rate':sum(r['changes'] for r in rows)/sum(r['comparisons'] for r in rows),'mean_purity':float(np.mean([r['purity'] for r in rows])),'mean_rmse':float(np.mean([r['position_rmse'] for r in rows]))}
summary=[summarize(val_rows,w) for w in WEIGHTS]
# Predeclared primary criterion; exact ties choose lower appearance weight.
selected=min(summary,key=lambda r:(r['change_rate'],r['weight']))
lock={'weight':selected['weight'],'criterion':'min pooled val change rate; ties smaller weight','val_seeds':VAL_SEEDS,'test_seeds':TEST_SEEDS,'gate':GATE,'n_init':N_INIT,'max_age':MAX_AGE,'unmatched':UNMATCHED}
(RESULTS/'selection_locked.json').write_text(json.dumps(lock,indent=2))
def write_csv(name,rows):
 with (RESULTS/name).open('w',newline='') as f:
  w=csv.DictWriter(f,fieldnames=rows[0]);w.writeheader();w.writerows(rows)
write_csv('validation.csv',val_rows);write_csv('validation_summary.csv',summary)
for row in summary:print(row)
print('LOCKED before test:',lock)
fig,axs=plt.subplots(1,2,figsize=(9,3),constrained_layout=True)
axs[0].plot(WEIGHTS,[r['change_rate'] for r in summary],'o-');axs[0].set(xlabel='appearance weight',ylabel='observed ID change rate')
axs[1].plot(WEIGHTS,[r['mean_purity'] for r in summary],'o-',label='purity');axs[1].set(xlabel='appearance weight',ylabel='mean track purity',ylim=(0,1.02))
for ax in axs:ax.grid()
fig.savefig(RESULTS/'validation.png');plt.show();plt.close(fig)

## Фиксированный стресс-тест на validation
После записи `selection_locked.json` поменяйте только appearance: в кадрах 18–29, когда оба объекта 0 и 1 наблюдаются, их дескрипторы меняются местами. `z`, рамки и GT остаются прежними. Не перенастраивайте weight, gate или lifecycle по этому результату. Заранее предскажите, какой вариант потеряет больше ID, затем сопоставьте изменения, фрагменты и ложные подтверждения на всех четырёх val seeds. `association_audit.json` сохраняет матрицы одного заранее выбранного val seed для ручного восстановления решения.

In [ ]:
# Secondary val-only diagnostic after the standard-val decision is locked.
lock=json.loads((RESULTS/'selection_locked.json').read_text())
stress_rows=[]
for seed in VAL_SEEDS:
 original=val_sequences[seed];altered=stress_appearance(original)
 assert all(np.array_equal(a[key],b[key]) for a,b in zip(original,altered)
            for key in ('z','boxes','gt_ids','true_positions'))
 for name,weight in [('motion',0.),('selected',lock['weight'])]:
  row,_=evaluate(altered,weight)
  stress_rows.append({'split':'val_stress','seed':seed,'method':name,**row})
write_csv('stress_validation.csv',stress_rows)
print('Val-only feature swap; weight remains locked:')
for row in stress_rows:print(row['seed'],row['method'],'changes',row['changes'],
                            'fragments',row['fragments'],'false_confirmed',row['false_confirmed'])

# One fixed validation sequence for hand reconstruction of a real assignment matrix.
audit_tracker=TeachingTracker(lock['weight']);audit=[]
for t,frame in enumerate(val_sequences[VAL_SEEDS[0]]):
 audit_tracker.step(frame['z'],frame['appearance'],frame['boxes'])
 audit.append({'frame':t,'gt_ids_for_analysis':frame['gt_ids'].tolist(),
               **audit_tracker.last_decision})
(RESULTS/'association_audit.json').write_text(json.dumps(audit,ensure_ascii=False,indent=2))
print('Association audit saved for fixed val seed',VAL_SEEDS[0],'; frames=',len(audit))

## Решение перед test
Линии на итоговом графике соединяют наблюдавшиеся точки и не обозначают измерения внутри пропуска.
**Заполните:** weight, val changes/comparisons, purity, RMSE, fragments/observed_gt_ids, false_confirmed/confirmed_tracks, результат stress и ограничение оценщика. По `association_audit.json` вручную восстановите хотя бы одно ненулевое назначение: track IDs, матрица cost, mask allowed, цена отказа и пары. Не меняйте Q, R, gate или seeds после просмотра test.

In [ ]:
lock=json.loads((RESULTS/'selection_locked.json').read_text())
test_rows=[];traces={}
for seed in TEST_SEEDS:
 sequence=make_sequence(seed)
 for name,weight in [('motion',0.),('selected',lock['weight'])]:
  row,trace=evaluate(sequence,weight);test_rows.append({'split':'test','seed':seed,'method':name,**row});traces[(seed,name)]=trace
write_csv('test.csv',test_rows)
for row in test_rows:print(row)
# Same first test sequence, all GT IDs, no cherry-picking.
fig,axs=plt.subplots(2,2,figsize=(11,6),constrained_layout=True)
for row,name in enumerate(['motion','selected']):
 trace=traces[(TEST_SEEDS[0],name)]
 for gid in range(3):
  part=[r for r in trace if r['gt_id']==gid];t=[r['frame'] for r in part]
  axs[row,0].plot(t,[r['track_id'] for r in part],'.-',label=f'GT {gid}')
  axs[row,1].plot([r['x'] for r in part],[r['y'] for r in part],'.-',label=f'GT {gid}')
 axs[row,0].set(title=name,xlabel='frame',ylabel='assigned track ID');axs[row,1].set(title='same scene: filtered positions',xlabel='x',ylabel='y')
 for ax in axs[row]:ax.grid();ax.legend()
fig.savefig(RESULTS/'same_sequence.png');plt.show();plt.close(fig)
flat=[{'seed':seed,'method':name,**r} for (seed,name),trace in traces.items() for r in trace]
write_csv('assignment_trace.csv',flat)

## Реальный видеопример: шумное измерение до трекера
OpenCV sample vtest: первые 60 кадров, уменьшенные до 384×288. MOG2 выделяет foreground-компоненты, затем их центры в единицах 16 пикселей получает тот же учебный трекер с weight=0 и одинаковыми нейтральными appearance-векторами. Размеры компонент сохраняются для рисунка, а трекер использует фиксированные proxy-рамки. Это **не детектор людей** и не полная SORT/DeepSORT реализация. Пропавший foreground не означает исчезнувший объект.

Перед запуском спрогнозируйте две ошибки такого измерения. После запуска укажите на трёх фиксированных кадрах конкретные номера component ID и track ID, минимум один разрыв или ложную связь. Разберите, что произошло с MOG2-входом и что решил трекер; объясните, какой сбой может исправить association, а какой требует улучшения detector. Нет ground truth, поэтому MOTA/IDF1/HOTA/recall здесь не считаются. Не используйте реальный клип для подбора weight основного опыта.

Если сеть недоступна, заранее сохраните исходное видео из указанной в ячейке ссылки; путь можно передать переменной окружения `COURSE_VIDEO_PATH`. SHA остается обязательной. Видео не включайте в сдаваемый архив/репозиторий.

In [ ]:
# Optional network only if the pinned sample is absent. File stays outside repository.
import cv2,urllib.request,hashlib,tempfile,os
VIDEO_URL='https://raw.githubusercontent.com/opencv/opencv/4.10.0/samples/data/vtest.avi'
VIDEO_SHA256='45cddc9490be69345cbdab64ca583be65987e864ca408038e648db99e10516cf'
cache=Path(tempfile.gettempdir())/'applied-ai-tracking';cache.mkdir(exist_ok=True)
video_path=Path(os.environ.get('COURSE_VIDEO_PATH',cache/'vtest.avi'))
if not video_path.exists():
 partial=video_path.with_suffix(video_path.suffix+'.part');digest=hashlib.sha256();received=0
 try:
  with urllib.request.urlopen(VIDEO_URL,timeout=60) as response,partial.open('wb') as target:
   total=int(response.headers.get('Content-Length') or 0)
   while True:
    chunk=response.read(1024*1024)
    if not chunk:break
    target.write(chunk);digest.update(chunk);received+=len(chunk)
    print(f'Видео: {received/1024/1024:.1f} MiB'+(f' / {total/1024/1024:.1f} MiB' if total else ''))
  if digest.hexdigest()!=VIDEO_SHA256:raise RuntimeError('SHA-256 downloaded video mismatch')
  partial.replace(video_path)
 except Exception:
  partial.unlink(missing_ok=True)
  raise
else:
 digest=hashlib.sha256()
 with video_path.open('rb') as source:
  for chunk in iter(lambda:source.read(1024*1024),b''):digest.update(chunk)
if digest.hexdigest()!=VIDEO_SHA256:raise RuntimeError('Wrong video file; do not silently use different observations')
cap=cv2.VideoCapture(str(video_path));assert cap.isOpened()
model=cv2.createBackgroundSubtractorMOG2(history=100,varThreshold=25,detectShadows=True)
chosen={};counts=[];video_rows=[];video_tracker=TeachingTracker(0.);start=time.perf_counter()
for t in range(60):
 ok,frame=cap.read();assert ok
 frame=cv2.resize(frame,(384,288));foreground=model.apply(frame)
 binary=np.uint8(foreground==255)*255
 binary=cv2.morphologyEx(binary,cv2.MORPH_OPEN,np.ones((3,3),np.uint8))
 contours,_=cv2.findContours(binary,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
 boxes=[cv2.boundingRect(c) for c in contours if cv2.contourArea(c)>=100]
 # The 4D teaching tracker uses component centres in 16-pixel units and fixed
 # proxy boxes. No appearance network or person detector is used here.
 if boxes:
  raw=np.asarray(boxes,float);centres=raw[:,:2]+raw[:,2:]/2
  z=(centres-np.array([192.,144.]))/16.
 else:z=np.empty((0,2))
 proxy=np.c_[z[:,0]-1.5,z[:,1]-2,z[:,0]+1.5,z[:,1]+2]
 neutral=np.tile(np.array([1.,0,0,0,0,0,0,0]),(len(z),1))
 assigned,_=video_tracker.step(z,neutral,proxy)
 status_by_id={tr['id']:tr['status'] for tr in video_tracker.tracks}
 for j,((x,y,w,h),tid) in enumerate(zip(boxes,assigned)):
  video_rows.append({'frame':t,'component':j,'x':x,'y':y,'w':w,'h':h,
                     'track_id':int(tid),'status':status_by_id[int(tid)]})
 counts.append({'frame':t,'components':len(boxes)})
 if t in [20,40,59]:chosen[t]=(frame.copy(),binary.copy(),boxes,assigned.tolist())
cap.release();elapsed=time.perf_counter()-start
fig,axs=plt.subplots(3,2,figsize=(10,9),constrained_layout=True)
for row,(t,(frame,mask,boxes,ids)) in enumerate(chosen.items()):
 axs[row,0].imshow(cv2.cvtColor(frame,cv2.COLOR_BGR2RGB));axs[row,1].imshow(mask,cmap='gray',vmin=0,vmax=255)
 for (x,y,w,h),tid in zip(boxes,ids):
  axs[row,0].add_patch(Rectangle((x,y),w,h,fill=False,color='lime',lw=1.4))
  axs[row,0].text(x,max(0,y-3),f'ID {tid}',color='yellow',fontsize=8,weight='bold')
 axs[row,0].set_title(f'frame {t}: component boxes and teaching IDs');axs[row,1].set_title('MOG2 foreground; shadows excluded')
 for ax in axs[row]:ax.axis('off')
fig.savefig(RESULTS/'real_video_foreground.png');plt.show();plt.close(fig)
write_csv('real_video_components.csv',counts)
write_csv('real_video_tracks.csv',video_rows)
(RESULTS/'real_video_run.json').write_text(json.dumps({'sha256':VIDEO_SHA256,'frames':60,'resize':[384,288],
 'seconds':elapsed,'opencv':cv2.__version__,'ground_truth':False,
 'tracking_input':'MOG2 component centres / 16 pixels; fixed proxy boxes',
 'appearance':'neutral equal vectors; motion-only weight=0','created_tracks':video_tracker.next_id},indent=2))
print(f'60 frames processed in {elapsed:.3f}s. IDs track foreground components only; no GT or MOT metrics.')

## Сдача и интерпретация
1. Гипотеза, семь реализованных TODO, собственные интеграционные тесты и таблица 12 опытов демо (по 2 на стенд).
2. Val/test/stress CSV, fixed config, `association_audit.json`, графики одинаковой последовательности и `assignment_trace.csv`.
3. Найдите один конкретный момент смены ID и один успешный переход через miss. Сопоставьте тот же GT ID/кадры у motion и selected. Какую часть результата объясняют признаки, а какую ограничивает motion gate?
4. Объясните, почему purity = 1 возможна при множестве фрагментов. Почему ваша continuity-метрика не заменяет MOTA/IDF1/HOTA? Какие ошибки detector она не штрафует?
5. Три наблюдения на реальных кадрах и `real_video_tracks.csv`: разберите конкретные ID, настройки MOG2 и отсутствие GT; не выдавайте число компонентов за число людей.

Стресс-тест appearance обязателен только на validation после фиксации выбора. Вывод об ухудшении либо отсутствии ухудшения подтверждайте конкретными seed/кадрами; test повторно для подбора не используйте.

Источники: [SORT](https://arxiv.org/abs/1602.00763), [DeepSORT](https://arxiv.org/abs/1703.07402), [OpenCV background subtraction](https://docs.opencv.org/4.x/d1/dc5/tutorial_background_subtraction.html).